# Superstore Sales Analysis - Full Assignment
### Subqueries, Correlated Subqueries, CTEs, Window Functions, JOINs, Aggregations, CASE, Date Functions

**Target dialect:** MySQL 8.0. All the queries in this notebook are written in valid MySQL
8.0 syntax and match what's in `superstore_analysis_mysql.sql`. This environment doesn't
have a MySQL server set up, so the outputs shown here were generated using SQLite loaded
with the same data instead - anywhere the syntax actually differs between the two is
noted directly. For the real results, run `superstore_analysis_mysql.sql` against an
actual MySQL 8.0 server.

**Dataset:** the Kaggle "Superstore Dataset Final" (`vivek468/superstore-dataset-final`),
uploaded directly for this project - 9,994 order-line rows, 793 unique customers, 1,862
unique products, and 5,009 distinct orders, covering dates from January 2014 to December
2017. Every result shown here comes from the actual dataset, not made-up data.


# Data Setup
## Explanation
Loads the real Kaggle CSV and applies three corrections the raw file needs before it can
be modeled cleanly (see `build_db_real.py` for the full loader):

1. **Encoding** - the file is Windows-1252/Latin-1, not UTF-8 (a handful of customer names
 contain German characters, e.g. "Roy Französisch").
2. **Dates** - stored as US-format text (`M/D/YYYY`, e.g. `11/8/2016`) → normalized to ISO
 `YYYY-MM-DD`.
3. **Postal codes** - 438 New England ZIP codes (CT/MA/RI/NH/ME/NJ) lost their leading zero
 somewhere upstream in this Kaggle release (e.g. Fairfield, CT is stored as `6824` instead
 of `06824`). Re-padded to 5 digits, since all US ZIP codes are 5 digits.

Then connects to the resulting `superstore.db` and confirms row counts.

In [1]:
import sqlite3
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

conn = sqlite3.connect("superstore.db")
for t in ["superstore_raw", "customers", "products", "orders"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t:15s}: {n} rows")

superstore_raw : 9994 rows
customers      : 793 rows
products       : 1862 rows
orders         : 9994 rows


## Output
Row counts confirm the load: `superstore_raw` holds one row per order line (9,994);
`customers` and `products` are deduplicated to one row per natural key; `orders` mirrors
`superstore_raw` at the same (order-line) grain, per the normalized schema in Part 1.

## Insight
Deduplicating with `SELECT DISTINCT` is what turns a flat, denormalized export into a
proper star-schema-style model - it's the step that makes every JOIN in this notebook
possible without double-counting a customer's descriptive attributes for every line
they ever ordered. Two more subtle data-quality issues surfaced while building this
schema and required a deliberate modeling decision each - see the next section.

# Data Quality Findings (found while building the schema)

Two issues in the real dataset would have silently broken a naive
`INSERT ... SELECT DISTINCT` (duplicate primary keys) or produced a misleading schema
(treating a variable attribute as if it were fixed). Both are documented here rather than
quietly patched, since they're the sort of thing a reviewer should be told about.

In [2]:
# 1) Ship-to geography is NOT fixed per customer -- the same customer_id ships to
#    different cities/states/regions across their orders.
geo_variability = pd.read_sql_query("""
    SELECT customer_id, COUNT(DISTINCT city) AS distinct_cities,
           COUNT(DISTINCT region) AS distinct_regions
    FROM superstore_raw
    GROUP BY customer_id
""", conn)
multi_city_customers = (geo_variability["distinct_cities"] > 1).sum()
print(f"Customers who shipped to more than one city: {multi_city_customers} of {len(geo_variability)}")
print("--> city/state/postal_code/region were modeled on `orders`, not `customers`.\n")

# 2) 32 product_ids map to two different product_name values in the source data.
name_variability = pd.read_sql_query("""
    SELECT product_id, COUNT(DISTINCT product_name) AS distinct_names
    FROM superstore_raw
    GROUP BY product_id
    HAVING COUNT(DISTINCT product_name) > 1
""", conn)
print(f"Product IDs with more than one Product Name: {len(name_variability)}")
example = pd.read_sql_query(f"""
    SELECT DISTINCT product_id, product_name FROM superstore_raw
    WHERE product_id = '{name_variability.iloc[0]["product_id"]}'
""", conn)
print("Example:")
print(example.to_string(index=False))
print("--> resolved by keeping one canonical (alphabetically-last) name per product_id.")

Customers who shipped to more than one city: 780 of 793
--> city/state/postal_code/region were modeled on `orders`, not `customers`.

Product IDs with more than one Product Name: 32
Example:
     product_id                                     product_name
FUR-BO-10002213            DMI Eclipse Executive Suite Bookcases
FUR-BO-10002213 Sauder Forest Hills Library, Woodland Oak Finish
--> resolved by keeping one canonical (alphabetically-last) name per product_id.


## Insight
Both issues are common in real-world "customer/product master data" and are exactly why a
senior analyst inspects functional dependencies (does this column truly depend only on the
key I think it depends on?) before finalizing a dimensional model, rather than trusting
column names at face value. Shipping to multiple cities is normal customer behavior, not
bad data - but it does mean "customer region" isn't a well-defined single value, only
"region of a given order" is.

# Query 2.1 - Orders with Sales Above Average

## Explanation
Concept demonstrated: Subquery (scalar, non-correlated)

Why this query: identifies high-value order lines relative to the whole business, useful for
spotting big-ticket transactions worth reviewing (fraud checks, VIP fulfillment, etc.).

Steps:
1. The inner query `SELECT AVG(sales) FROM orders` runs once and returns a single number.
2. The outer query scans `orders`, keeping only rows where `sales` exceeds that number.
3. A `JOIN` to `customers` adds the human-readable customer name to each qualifying row.

Time complexity: the scalar subquery is `O(n)` (one pass to average). The outer filter
is `O(n)` with a full scan, or `O(log n + k)` per lookup if `sales` were indexed (rarely
worth indexing a filtered numeric column here) - overall effectively `O(n)`.

In [3]:
mysql_sql = """
SELECT
    o.order_id      AS order_id,
    c.customer_name AS customer_name,
    o.product_id    AS product_id,
    o.sales         AS sales
FROM orders AS o
INNER JOIN customers AS c
    ON c.customer_id = o.customer_id
WHERE o.sales > (
    SELECT AVG(sales) FROM orders
)
ORDER BY o.sales DESC;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
df = pd.read_sql_query(mysql_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.head(10).to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
SELECT
    o.order_id      AS order_id,
    c.customer_name AS customer_name,
    o.product_id    AS product_id,
    o.sales         AS sales
FROM orders AS o
INNER JOIN customers AS c
    ON c.customer_id = o.customer_id
WHERE o.sales > (
    SELECT AVG(sales) FROM orders
)
ORDER BY o.sales DESC;


Rows returned: 2360

      order_id      customer_name      product_id     sales
CA-2014-145317        Sean Miller TEC-MA-10002412 22638.480
CA-2016-118689       Tamara Chand TEC-CO-10004722 17499.950
CA-2017-140151       Raymond Buch TEC-CO-10004722 13999.960
CA-2017-127180       Tom Ashbrook TEC-CO-10004722 11199.968
CA-2017-166709       Hunter Lopez TEC-CO-10004722 10499.970
CA-2016-117121      Adrian Barton OFF-BI-10000545  9892.740
CA-2014-116904       Sanjit Chand OFF-BI-10001120  9449.950
US-2016-107440       Bill Shonely TEC-MA-10001047  9099.930
CA-2016-158841       Sanjit Engle TEC-MA-10001127  8749.950
CA-2016

## Insight
Only a minority of order lines clear the average - sales values are right-skewed
(a handful of very large orders pull the mean well above the typical order), which is
common in retail and is revisited in Query 9.5's High/Medium/Low segmentation.

# Query 2.2 - Highest Sales Order per Customer

## Explanation
Concept demonstrated: Correlated Subquery

Why this query: answers "what was each customer's single biggest purchase line?" - a common
account-management question.

Steps:
1. For every row `o` in the outer query, the inner subquery re-runs, restricted to
 `o2.customer_id = o.customer_id` (this is what makes it *correlated* - it depends on
 the outer row).
2. It returns that customer's maximum `sales` value.
3. The outer `WHERE` keeps only the row(s) whose `sales` equals that maximum.

Time complexity: naively `O(n²)` (the correlated subquery re-scans `orders` for every
outer row), but with an index on `orders(customer_id)` each correlated lookup becomes
`O(log n)`, bringing the total to roughly `O(n log n)` - this is why
`idx_orders_customer` was created in Part 1.

In [4]:
mysql_sql = """
SELECT
    c.customer_name AS customer_name,
    o.order_id      AS order_id,
    o.product_id    AS product_id,
    o.sales         AS sales
FROM orders AS o
INNER JOIN customers AS c
    ON c.customer_id = o.customer_id
WHERE o.sales = (
    SELECT MAX(o2.sales)
    FROM orders AS o2
    WHERE o2.customer_id = o.customer_id
)
ORDER BY o.sales DESC;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
df = pd.read_sql_query(mysql_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.head(10).to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
SELECT
    c.customer_name AS customer_name,
    o.order_id      AS order_id,
    o.product_id    AS product_id,
    o.sales         AS sales
FROM orders AS o
INNER JOIN customers AS c
    ON c.customer_id = o.customer_id
WHERE o.sales = (
    SELECT MAX(o2.sales)
    FROM orders AS o2
    WHERE o2.customer_id = o.customer_id
)
ORDER BY o.sales DESC;


Rows returned: 795

     customer_name       order_id      product_id     sales
       Sean Miller CA-2014-145317 TEC-MA-10002412 22638.480
      Tamara Chand CA-2016-118689 TEC-CO-10004722 17499.950
      Raymond Buch CA-2017-140151 TEC-CO-10004722 13999.960
      Tom Ashbrook CA-2017-127180 TEC-CO-10004722 11199.968
      Hunter Lopez CA-2017-166709 TEC-CO-10004722 10499.970
     Adrian Barton CA-2016-117121 OFF-BI-10000545  9892.740
      Sanjit Chand CA-2014-116904 OFF-BI-10001120  9449.950
      Bill Shonely US-2016-107440 TEC-MA-10001047  9099.930
      Sanjit E

## Insight
Line-level "highest sales" isn't always the same as a customer's biggest *order*
(an order can span several lines) - Query 4.5 answers the order-level version of this
question and the two can disagree for customers whose biggest order had multiple items.

# Query 2.3 - Total Sales per Customer

## Explanation
Concept demonstrated: CTE (Common Table Expression)

Why this query: the foundational aggregation nearly every later query builds on (rankings,
above-average filters, top/bottom N all start here).

Steps:
1. The `WITH customer_totals AS (...)` CTE groups `orders` by `customer_id` and sums
 `sales`, materializing a small (one row per customer) named result set.
2. The outer query joins that CTE back to `customers` to attach the display name.
3. Results are sorted descending by total sales.

Time complexity: `O(n)` to scan and aggregate `orders` (or `O(n log n)` if MySQL
chooses a sort-based `GROUP BY` instead of a hash aggregate), plus `O(m log m)` for the
final sort over `m` customers.

In [5]:
mysql_sql = """
WITH customer_totals AS (
    SELECT
        customer_id,
        SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT
    c.customer_name          AS customer_name,
    ROUND(ct.total_sales, 2) AS total_sales
FROM customer_totals AS ct
INNER JOIN customers AS c
    ON c.customer_id = ct.customer_id
ORDER BY ct.total_sales DESC;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
df = pd.read_sql_query(mysql_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.head(10).to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
WITH customer_totals AS (
    SELECT
        customer_id,
        SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT
    c.customer_name          AS customer_name,
    ROUND(ct.total_sales, 2) AS total_sales
FROM customer_totals AS ct
INNER JOIN customers AS c
    ON c.customer_id = ct.customer_id
ORDER BY ct.total_sales DESC;


Rows returned: 793

     customer_name  total_sales
       Sean Miller     25043.05
      Tamara Chand     19052.22
      Raymond Buch     15117.34
      Tom Ashbrook     14595.62
     Adrian Barton     14473.57
      Ken Lonsdale     14175.23
      Sanjit Chand     14142.33
      Hunter Lopez     12873.30
      Sanjit Engle     12209.44
Christopher Conant     12129.07


## Insight
Total sales per customer is the single most reused metric in this assignment -
every ranking, threshold, and top/bottom question downstream is a transformation of this
one CTE, which is why CTEs are worth learning: define the aggregation once, reuse
it everywhere.

# Query 2.4 - Customers with Above-Average Total Sales

## Explanation
Concept demonstrated: CTE + Subquery

Why this query: segments the customer base into "above-average" vs. the rest - a standard
input to loyalty-tier or account-priority logic.

Steps:
1. Same `customer_totals` CTE as Query 2.3.
2. A scalar subquery, `SELECT AVG(total_sales) FROM customer_totals`, averages the
 *per-customer totals* (not raw order lines - an important distinction, see Insight).
3. The outer query keeps only customers whose total exceeds that average.

Time complexity: `O(n)` to build the CTE, `O(m)` to average it, `O(m)` to filter -
linear overall in the number of order lines `n` and customers `m`.

In [6]:
mysql_sql = """
WITH customer_totals AS (
    SELECT
        customer_id,
        SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT
    c.customer_name          AS customer_name,
    ROUND(ct.total_sales, 2) AS total_sales
FROM customer_totals AS ct
INNER JOIN customers AS c
    ON c.customer_id = ct.customer_id
WHERE ct.total_sales > (
    SELECT AVG(total_sales) FROM customer_totals
)
ORDER BY ct.total_sales DESC;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
df = pd.read_sql_query(mysql_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
WITH customer_totals AS (
    SELECT
        customer_id,
        SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT
    c.customer_name          AS customer_name,
    ROUND(ct.total_sales, 2) AS total_sales
FROM customer_totals AS ct
INNER JOIN customers AS c
    ON c.customer_id = ct.customer_id
WHERE ct.total_sales > (
    SELECT AVG(total_sales) FROM customer_totals
)
ORDER BY ct.total_sales DESC;


Rows returned: 294

       customer_name  total_sales
         Sean Miller     25043.05
        Tamara Chand     19052.22
        Raymond Buch     15117.34
        Tom Ashbrook     14595.62
       Adrian Barton     14473.57
        Ken Lonsdale     14175.23
        Sanjit Chand     14142.33
        Hunter Lopez     12873.30
        Sanjit Engle     12209.44
  Christopher Conant     12129.07
        Todd Sumrall     11891.75
           Greg Tran     11820.12
        Becky Martin     11789.63
 

## Insight
This average is computed over *customer totals*, not over individual order
lines - averaging line-level `sales` (like Query 2.1 does) and averaging per-customer
`SUM(sales)` are different numbers with different meanings, and mixing them up is a common
analyst mistake.

# Query 2.5 - Rank Customers by Total Sales

## Explanation
Concept demonstrated: Window Functions: RANK() / DENSE_RANK()

Why this query: produces a full sales leaderboard, and shows the two common ways SQL
handles ties in a ranking.

Steps:
1. Build `customer_totals` as before.
2. `RANK() OVER (ORDER BY total_sales DESC)` assigns 1, 2, 3, ... but **skips** numbers
 after a tie (two customers tied for 2nd both get 2, the next gets 4).
3. `DENSE_RANK()` assigns 1, 2, 3, ... with **no gap** after a tie (two tied for 2nd both
 get 2, the next gets 3).

Time complexity: `O(m log m)` - the window function requires sorting the `m` customer
rows by `total_sales` once; `RANK`/`DENSE_RANK` are then a single linear pass over the
sorted set.

In [7]:
mysql_sql = """
WITH customer_totals AS (
    SELECT
        customer_id,
        SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT
    c.customer_name                                            AS customer_name,
    ROUND(ct.total_sales, 2)                                    AS total_sales,
    RANK()       OVER (ORDER BY ct.total_sales DESC)            AS sales_rank,
    DENSE_RANK() OVER (ORDER BY ct.total_sales DESC)            AS sales_dense_rank
FROM customer_totals AS ct
INNER JOIN customers AS c
    ON c.customer_id = ct.customer_id
ORDER BY sales_rank;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
df = pd.read_sql_query(mysql_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.head(15).to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
WITH customer_totals AS (
    SELECT
        customer_id,
        SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT
    c.customer_name                                            AS customer_name,
    ROUND(ct.total_sales, 2)                                    AS total_sales,
    RANK()       OVER (ORDER BY ct.total_sales DESC)            AS sales_rank,
    DENSE_RANK() OVER (ORDER BY ct.total_sales DESC)            AS sales_dense_rank
FROM customer_totals AS ct
INNER JOIN customers AS c
    ON c.customer_id = ct.customer_id
ORDER BY sales_rank;


Rows returned: 793

     customer_name  total_sales  sales_rank  sales_dense_rank
       Sean Miller     25043.05           1                 1
      Tamara Chand     19052.22           2                 2
      Raymond Buch     15117.34           3                 3
      Tom Ashbrook     14595.62           4                 4
     Adrian Barton 

## Insight
In this dataset customer totals are all distinct floating-point sums, so
`RANK()` and `DENSE_RANK()` happen to agree here - but they will diverge the moment two
customers land on the exact same total, which is why picking the right one still matters
(`RANK()` for "true leaderboard position," `DENSE_RANK()` for "how many distinct tiers
exist").

# Query 2.6 - Row Number per Order within Customer

## Explanation
Concept demonstrated: Window Function: ROW_NUMBER() + PARTITION BY

Why this query: numbers each customer's orders in chronological sequence (1st order, 2nd
order, ...) - useful for cohort/lifecycle analysis ("what did customers buy on their 3rd
order?").

Steps:
1. `PARTITION BY o.customer_id` resets the counter to 1 at the start of every new
 customer's rows.
2. `ORDER BY o.order_date, o.row_id` fixes the order within each partition (the
 `row_id` tiebreaker keeps same-day orders deterministic).
3. `ROW_NUMBER()` then assigns 1, 2, 3, ... with **no ties ever** - every row gets a
 unique number even if `sales` or dates are identical.

Time complexity: `O(n log n)` to sort `n` order-line rows into partitions, then a
linear pass to number them.

In [8]:
mysql_sql = """
SELECT
    c.customer_name AS customer_name,
    o.order_id      AS order_id,
    o.order_date    AS order_date,
    o.sales         AS sales,
    ROW_NUMBER() OVER (
        PARTITION BY o.customer_id
        ORDER BY o.order_date, o.row_id
    ) AS order_seq
FROM orders AS o
INNER JOIN customers AS c
    ON c.customer_id = o.customer_id
ORDER BY c.customer_name, order_seq;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
df = pd.read_sql_query(mysql_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.head(15).to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
SELECT
    c.customer_name AS customer_name,
    o.order_id      AS order_id,
    o.order_date    AS order_date,
    o.sales         AS sales,
    ROW_NUMBER() OVER (
        PARTITION BY o.customer_id
        ORDER BY o.order_date, o.row_id
    ) AS order_seq
FROM orders AS o
INNER JOIN customers AS c
    ON c.customer_id = o.customer_id
ORDER BY c.customer_name, order_seq;


Rows returned: 9994

customer_name       order_id order_date   sales  order_seq
Aaron Bergman CA-2014-152905 2014-02-18  12.624          1
Aaron Bergman CA-2014-156587 2014-03-07  48.712          2
Aaron Bergman CA-2014-156587 2014-03-07  17.940          3
Aaron Bergman CA-2014-156587 2014-03-07 242.940          4
Aaron Bergman CA-2016-140935 2016-11-10 221.980          5
Aaron Bergman CA-2016-140935 2016-11-10 341.960          6
Aaron Hawkins CA-2014-122070 2014-04-22 247.840          1
Aaron Hawkins CA-2014-122070 2014-04-22   9.912         

## Insight
Unlike `RANK()`, `ROW_NUMBER()` never produces a tie - that's precisely why it's
the right tool here: an order sequence number must be unique per customer even if two
orders happen to have identical dates.

# Query 2.7 - Top 3 Customers by Total Sales

## Explanation
Concept demonstrated: Window Function (RANK filtered via CTE)

Why this query: a window function's result can't be filtered in the same `SELECT`'s
`WHERE` clause (window functions evaluate after `WHERE`), so the rank has to be computed
in one query level (here, a second CTE) and filtered in the next.

Steps:
1. `customer_totals` aggregates sales per customer.
2. `ranked_customers` wraps that in `RANK()`.
3. The final `SELECT` filters `sales_rank <= 3` - this only works because the rank was
 computed in an earlier query stage.

Time complexity: same as Query 2.5, `O(m log m)` for the sort/rank, plus `O(m)` to
filter down to the top 3.

In [9]:
mysql_sql = """
WITH customer_totals AS (
    SELECT
        customer_id,
        SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
),
ranked_customers AS (
    SELECT
        c.customer_name           AS customer_name,
        ROUND(ct.total_sales, 2)  AS total_sales,
        RANK() OVER (ORDER BY ct.total_sales DESC) AS sales_rank
    FROM customer_totals AS ct
    INNER JOIN customers AS c
        ON c.customer_id = ct.customer_id
)
SELECT *
FROM ranked_customers
WHERE sales_rank <= 3
ORDER BY sales_rank;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
df = pd.read_sql_query(mysql_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
WITH customer_totals AS (
    SELECT
        customer_id,
        SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
),
ranked_customers AS (
    SELECT
        c.customer_name           AS customer_name,
        ROUND(ct.total_sales, 2)  AS total_sales,
        RANK() OVER (ORDER BY ct.total_sales DESC) AS sales_rank
    FROM customer_totals AS ct
    INNER JOIN customers AS c
        ON c.customer_id = ct.customer_id
)
SELECT *
FROM ranked_customers
WHERE sales_rank <= 3
ORDER BY sales_rank;


Rows returned: 3

customer_name  total_sales  sales_rank
  Sean Miller     25043.05           1
 Tamara Chand     19052.22           2
 Raymond Buch     15117.34           3


## Insight
`LIMIT 3` alone would *not* be reliable here if ties existed at the boundary
(it would arbitrarily cut a tied customer) - filtering on `sales_rank <= 3` guarantees
every customer tied for 3rd place is included.

# Final Combined Query - JOIN + CTE + Window Function
## Explanation
Concept demonstrated: all three techniques used together in one query.

This is the report the mini-project questions are largely derived from: one CTE
aggregates sales via a `JOIN` between `customers` and `orders`, and a window function
ranks the result - the canonical "customer leaderboard" query.

In [10]:
mysql_sql = """
WITH customer_sales AS (
    SELECT
        c.customer_id                AS customer_id,
        c.customer_name              AS customer_name,
        SUM(o.sales)                 AS total_sales
    FROM customers AS c
    INNER JOIN orders AS o
        ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.customer_name
)
SELECT
    customer_name             AS `Customer Name`,
    ROUND(total_sales, 2)     AS `Total Sales`,
    RANK() OVER (ORDER BY total_sales DESC) AS `Customer Rank`
FROM customer_sales
ORDER BY `Customer Rank`;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
df = pd.read_sql_query(mysql_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.head(15).to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
WITH customer_sales AS (
    SELECT
        c.customer_id                AS customer_id,
        c.customer_name              AS customer_name,
        SUM(o.sales)                 AS total_sales
    FROM customers AS c
    INNER JOIN orders AS o
        ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.customer_name
)
SELECT
    customer_name             AS `Customer Name`,
    ROUND(total_sales, 2)     AS `Total Sales`,
    RANK() OVER (ORDER BY total_sales DESC) AS `Customer Rank`
FROM customer_sales
ORDER BY `Customer Rank`;


Rows returned: 793

     Customer Name  Total Sales  Customer Rank
       Sean Miller     25043.05              1
      Tamara Chand     19052.22              2
      Raymond Buch     15117.34              3
      Tom Ashbrook     14595.62              4
     Adrian Barton     14473.57              5
      Ken Lonsdale     14175.23              6
      Sanjit Chand     14142.3

## Insight
This single query is the "source of truth" leaderboard: every mini-project ranking
question (top 5, bottom 5, above-average) is a slice or re-sort of exactly this result
set, which is good practice - compute the core metric once, answer many questions from it.

# Mini Project - Customer Sales Insights

# Query MP.1 - Top 5 Customers

## Explanation
Concept demonstrated: Window Function + LIMIT

Why this query: identifies the accounts generating the most revenue - the first place any
account-management or retention effort should look.

Expected output columns: `customer_name`, `total_sales`, `sales_rank`.

Steps: aggregate sales per customer, rank descending, keep the top 5 rows.
Time complexity: `O(m log m)` to sort/rank `m` customers.

In [11]:
mysql_sql = """
WITH customer_sales AS (
    SELECT c.customer_id, c.customer_name, SUM(o.sales) AS total_sales
    FROM customers AS c
    INNER JOIN orders AS o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.customer_name
)
SELECT
    customer_name             AS customer_name,
    ROUND(total_sales, 2)     AS total_sales,
    RANK() OVER (ORDER BY total_sales DESC) AS sales_rank
FROM customer_sales
ORDER BY total_sales DESC
LIMIT 5;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
df = pd.read_sql_query(mysql_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
WITH customer_sales AS (
    SELECT c.customer_id, c.customer_name, SUM(o.sales) AS total_sales
    FROM customers AS c
    INNER JOIN orders AS o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.customer_name
)
SELECT
    customer_name             AS customer_name,
    ROUND(total_sales, 2)     AS total_sales,
    RANK() OVER (ORDER BY total_sales DESC) AS sales_rank
FROM customer_sales
ORDER BY total_sales DESC
LIMIT 5;


Rows returned: 5

customer_name  total_sales  sales_rank
  Sean Miller     25043.05           1
 Tamara Chand     19052.22           2
 Raymond Buch     15117.34           3
 Tom Ashbrook     14595.62           4
Adrian Barton     14473.57           5


## Insight
These five customers represent the business's most valuable relationships -
losing any one of them would have an outsized revenue impact, so they're strong
candidates for a dedicated account manager or loyalty-tier treatment.

# Query MP.2 - Bottom 5 Customers

## Explanation
Concept demonstrated: Window Function (ascending)

Why this query: flags the weakest active accounts - candidates for a re-engagement
campaign or a review of whether they're worth the servicing cost.

Expected output columns: `customer_name`, `total_sales`, `rank_from_bottom`.

Steps: identical to MP.1 but sorted ascending instead of descending.
Time complexity: `O(m log m)`.

In [12]:
mysql_sql = """
WITH customer_sales AS (
    SELECT c.customer_id, c.customer_name, SUM(o.sales) AS total_sales
    FROM customers AS c
    INNER JOIN orders AS o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.customer_name
)
SELECT
    customer_name             AS customer_name,
    ROUND(total_sales, 2)     AS total_sales,
    RANK() OVER (ORDER BY total_sales ASC) AS rank_from_bottom
FROM customer_sales
ORDER BY total_sales ASC
LIMIT 5;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
df = pd.read_sql_query(mysql_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
WITH customer_sales AS (
    SELECT c.customer_id, c.customer_name, SUM(o.sales) AS total_sales
    FROM customers AS c
    INNER JOIN orders AS o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.customer_name
)
SELECT
    customer_name             AS customer_name,
    ROUND(total_sales, 2)     AS total_sales,
    RANK() OVER (ORDER BY total_sales ASC) AS rank_from_bottom
FROM customer_sales
ORDER BY total_sales ASC
LIMIT 5;


Rows returned: 5

  customer_name  total_sales  rank_from_bottom
  Thais Sissman         4.83                 1
   Lela Donovan         5.30                 2
   Carl Jackson        16.52                 3
Mitch Gastineau        16.74                 4
     Roy Skaria        22.33                 5


## Insight
Low total sales alone doesn't mean a weak relationship - cross-reference with
MP.3 (single-order customers) below, since a customer here might simply be new rather
than low-value.

# Query MP.3 - Customers Who Made Only One Order

## Explanation
Concept demonstrated: GROUP BY + HAVING

Why this query: finds "one-and-done" customers - acquired once but never retained,
useful for win-back campaign targeting.

Expected output columns: `customer_name`, `order_count`.

Steps: group order lines by customer, count *distinct* order IDs (not lines -
one order can have several lines), keep groups where that count is exactly 1.
Time complexity: `O(n)` to group `n` order lines, `O(m)` to filter `m` groups.

In [13]:
mysql_sql = """
SELECT
    c.customer_name                    AS customer_name,
    COUNT(DISTINCT o.order_id)         AS order_count
FROM customers AS c
INNER JOIN orders AS o
    ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.customer_name
HAVING COUNT(DISTINCT o.order_id) = 1
ORDER BY c.customer_name;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
df = pd.read_sql_query(mysql_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.head(20).to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
SELECT
    c.customer_name                    AS customer_name,
    COUNT(DISTINCT o.order_id)         AS order_count
FROM customers AS c
INNER JOIN orders AS o
    ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.customer_name
HAVING COUNT(DISTINCT o.order_id) = 1
ORDER BY c.customer_name;


Rows returned: 12

    customer_name  order_count
   Anemone Ratner            1
Anthony O'Donnell            1
     Carl Jackson            1
     Jenna Caffey            1
   Jocasta Rupert            1
     Lela Donovan            1
  Mitch Gastineau            1
Patricia Hirasaki            1
  Ricardo Emerson            1
    Roland Murray            1
Susan MacKendrick            1
    Theresa Coyne            1


## Insight
A meaningful share of the customer base has never placed a second order -
since acquiring them already cost a sales/marketing cycle, a targeted win-back offer is
usually cheaper than acquiring a brand-new customer of equivalent value.

# Query MP.4 - Customers with Above-Average Sales

## Explanation
Concept demonstrated: CTE + Subquery (same as 2.4)

Why this query: restated from Query 2.4 as a standalone mini-project deliverable.

Expected output columns: `customer_name`, `total_sales`.

**Step-by-step / complexity:** see Query 2.4 above.

In [14]:
mysql_sql = """
WITH customer_totals AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT
    c.customer_name           AS customer_name,
    ROUND(ct.total_sales, 2)  AS total_sales
FROM customer_totals AS ct
INNER JOIN customers AS c
    ON c.customer_id = ct.customer_id
WHERE ct.total_sales > (SELECT AVG(total_sales) FROM customer_totals)
ORDER BY ct.total_sales DESC;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
df = pd.read_sql_query(mysql_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.head(10).to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
WITH customer_totals AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT
    c.customer_name           AS customer_name,
    ROUND(ct.total_sales, 2)  AS total_sales
FROM customer_totals AS ct
INNER JOIN customers AS c
    ON c.customer_id = ct.customer_id
WHERE ct.total_sales > (SELECT AVG(total_sales) FROM customer_totals)
ORDER BY ct.total_sales DESC;


Rows returned: 294

     customer_name  total_sales
       Sean Miller     25043.05
      Tamara Chand     19052.22
      Raymond Buch     15117.34
      Tom Ashbrook     14595.62
     Adrian Barton     14473.57
      Ken Lonsdale     14175.23
      Sanjit Chand     14142.33
      Hunter Lopez     12873.30
      Sanjit Engle     12209.44
Christopher Conant     12129.07


## Insight
Because customer revenue is right-skewed (a few very large accounts pull the
mean upward), noticeably fewer than half of all customers clear the average - a reminder
that mean-based segmentation should usually be paired with a median or quartile check.

# Query MP.5 - Highest Order Value per Customer

## Explanation
Concept demonstrated: Correlated Subquery (order-level rollup)

Why this query: answers "what was each customer's single biggest *order* (not just
order line)?" - the business-accurate version of Query 2.2, since a real order can bundle
several products.

Expected output columns: `customer_name`, `order_id`, `highest_order_value`.

Steps:
1. `order_totals` first rolls order *lines* up to the *order* level with `SUM(sales)`.
2. A correlated subquery then finds each customer's maximum order-level total.
3. The outer query keeps the order(s) matching that maximum.

Time complexity: `O(n)` to build `order_totals` from `n` lines, then the correlated
max lookup is `O(k log k)` per customer over their `k` orders (or `O(k)` with an index),
totaling roughly `O(n + m·k)` in the worst case, `O(n)` with proper indexing.

In [15]:
mysql_sql = """
WITH order_totals AS (
    SELECT customer_id, order_id, SUM(sales) AS order_value
    FROM orders
    GROUP BY customer_id, order_id
)
SELECT
    c.customer_name                AS customer_name,
    ot.order_id                    AS order_id,
    ROUND(ot.order_value, 2)       AS highest_order_value
FROM order_totals AS ot
INNER JOIN customers AS c
    ON c.customer_id = ot.customer_id
WHERE ot.order_value = (
    SELECT MAX(ot2.order_value)
    FROM order_totals AS ot2
    WHERE ot2.customer_id = ot.customer_id
)
ORDER BY highest_order_value DESC;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
df = pd.read_sql_query(mysql_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.head(10).to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
WITH order_totals AS (
    SELECT customer_id, order_id, SUM(sales) AS order_value
    FROM orders
    GROUP BY customer_id, order_id
)
SELECT
    c.customer_name                AS customer_name,
    ot.order_id                    AS order_id,
    ROUND(ot.order_value, 2)       AS highest_order_value
FROM order_totals AS ot
INNER JOIN customers AS c
    ON c.customer_id = ot.customer_id
WHERE ot.order_value = (
    SELECT MAX(ot2.order_value)
    FROM order_totals AS ot2
    WHERE ot2.customer_id = ot.customer_id
)
ORDER BY highest_order_value DESC;


Rows returned: 793

customer_name       order_id  highest_order_value
  Sean Miller CA-2014-145317             23661.23
 Tamara Chand CA-2016-118689             18336.74
 Raymond Buch CA-2017-140151             14052.48
 Tom Ashbrook CA-2017-127180             13716.46
 Becky Martin CA-2014-139892             10539.90
 Hunter Lopez CA-2017-166709             10499.97
 

## Insight
Comparing this to Query 2.2 shows the two views can disagree: a customer's
single biggest *line item* is not always inside their single biggest *order* once
multi-item orders are rolled up - always clarify "order" vs. "order line" before reporting
a "biggest purchase" number to stakeholders.

# Bonus Analysis - 10 Advanced Business Questions

# Query 9.1 - Monthly Sales Trend

## Explanation
Concept demonstrated: Date Functions + Aggregation

Why this query: the first question in almost any sales analysis - is revenue trending up,
down, or seasonal?

Steps: truncate each `order_date` to its year-month, group by that value, sum
sales and count distinct orders per month.
Time complexity: `O(n)` to compute the month key and aggregate `n` rows.

In [16]:
mysql_sql = """
SELECT
    DATE_FORMAT(order_date, '%Y-%m')  AS sales_month,
    ROUND(SUM(sales), 2)              AS monthly_sales,
    COUNT(DISTINCT order_id)          AS order_count
FROM orders
GROUP BY DATE_FORMAT(order_date, '%Y-%m')
ORDER BY sales_month;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
print("[Executed via an equivalent SQLite-compatible query in this sandbox notebook -- see note below]")
exec_sql = """
SELECT
    strftime('%Y-%m', order_date)     AS sales_month,
    ROUND(SUM(sales), 2)              AS monthly_sales,
    COUNT(DISTINCT order_id)          AS order_count
FROM orders
GROUP BY strftime('%Y-%m', order_date)
ORDER BY sales_month;
"""
df = pd.read_sql_query(exec_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.head(15).to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
SELECT
    DATE_FORMAT(order_date, '%Y-%m')  AS sales_month,
    ROUND(SUM(sales), 2)              AS monthly_sales,
    COUNT(DISTINCT order_id)          AS order_count
FROM orders
GROUP BY DATE_FORMAT(order_date, '%Y-%m')
ORDER BY sales_month;

[Executed via an equivalent SQLite-compatible query in this sandbox notebook -- see note below]

Rows returned: 48

sales_month  monthly_sales  order_count
    2014-01       14236.90           32
    2014-02        4519.89           28
    2014-03       55691.01           71
    2014-04       28295.35           66
    2014-05       23648.29           69
    2014-06       34595.13           66
    2014-07       33946.39           65
    2014-08       27909.47           72
    2014-09       81777.35          130
    2014-10       31453.39           78
    2014-11       78628.72          151
    2014-12       69545.62          141
    2015-01       18174.08           29
    20

> **Dialect note:** MySQL's `DATE_FORMAT(date, '%Y-%m')` becomes SQLite's `strftime('%Y-%m', date)` - same output, different function name.

## Insight
Monthly totals set up Query 9.2's growth-rate calculation - always look at the
trend line, not just a single period's total, before concluding sales are up or down.

# Query 9.2 - Month-over-Month Sales Growth %

## Explanation
Concept demonstrated: Window Function: LAG()

Why this query: raw monthly totals don't say whether the business is accelerating or
decelerating - the percentage change month-to-month does.

Steps:
1. Aggregate monthly sales (as in 9.1) inside a CTE.
2. `LAG(total_sales) OVER (ORDER BY sales_month)` looks one row *back* in month order,
 returning the previous month's total on the current row.
3. Compute `% change = (current - previous) / previous * 100`.

Time complexity: `O(n)` to build the monthly CTE, `O(k log k)` to order `k` months for
the window function (small - dozens of rows at most).

In [17]:
mysql_sql = """
WITH monthly_sales AS (
    SELECT
        DATE_FORMAT(order_date, '%Y-%m') AS sales_month,
        SUM(sales)                       AS total_sales
    FROM orders
    GROUP BY DATE_FORMAT(order_date, '%Y-%m')
)
SELECT
    sales_month,
    ROUND(total_sales, 2)                                  AS total_sales,
    ROUND(LAG(total_sales) OVER (ORDER BY sales_month), 2) AS prev_month_sales,
    ROUND(
        100.0 * (total_sales - LAG(total_sales) OVER (ORDER BY sales_month))
        / LAG(total_sales) OVER (ORDER BY sales_month), 2
    ) AS mom_growth_pct
FROM monthly_sales
ORDER BY sales_month;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
print("[Executed via an equivalent SQLite-compatible query in this sandbox notebook -- see note below]")
exec_sql = """
WITH monthly_sales AS (
    SELECT
        strftime('%Y-%m', order_date)    AS sales_month,
        SUM(sales)                       AS total_sales
    FROM orders
    GROUP BY strftime('%Y-%m', order_date)
)
SELECT
    sales_month,
    ROUND(total_sales, 2)                                  AS total_sales,
    ROUND(LAG(total_sales) OVER (ORDER BY sales_month), 2) AS prev_month_sales,
    ROUND(
        100.0 * (total_sales - LAG(total_sales) OVER (ORDER BY sales_month))
        / LAG(total_sales) OVER (ORDER BY sales_month), 2
    ) AS mom_growth_pct
FROM monthly_sales
ORDER BY sales_month;
"""
df = pd.read_sql_query(exec_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.head(15).to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
WITH monthly_sales AS (
    SELECT
        DATE_FORMAT(order_date, '%Y-%m') AS sales_month,
        SUM(sales)                       AS total_sales
    FROM orders
    GROUP BY DATE_FORMAT(order_date, '%Y-%m')
)
SELECT
    sales_month,
    ROUND(total_sales, 2)                                  AS total_sales,
    ROUND(LAG(total_sales) OVER (ORDER BY sales_month), 2) AS prev_month_sales,
    ROUND(
        100.0 * (total_sales - LAG(total_sales) OVER (ORDER BY sales_month))
        / LAG(total_sales) OVER (ORDER BY sales_month), 2
    ) AS mom_growth_pct
FROM monthly_sales
ORDER BY sales_month;

[Executed via an equivalent SQLite-compatible query in this sandbox notebook -- see note below]

Rows returned: 48

sales_month  total_sales  prev_month_sales  mom_growth_pct
    2014-01     14236.90               NaN             NaN
    2014-02      4519.89          14236.90          -68.25
    2014-03     55691.01         

> **Dialect note:** Same `DATE_FORMAT` → `strftime` substitution as Query 9.1; `LAG()` itself is identical in both engines.

## Insight
The first month in the series has no prior month, so `mom_growth_pct` is
`NULL` there by design - that's expected, not a bug, and downstream reporting should
handle it explicitly rather than treating it as zero growth.

# Query 9.3 - Segment-wise Total Sales and Rank

## Explanation
Concept demonstrated: JOIN + Aggregation + RANK

Why this query: the Superstore segments (Consumer / Corporate / Home Office) are a
standard slice for go-to-market strategy - which segment should get the marketing budget?

Steps: join orders to customers for the segment attribute, sum sales per
segment, rank the (typically 3) segments.
Time complexity: `O(n)` join + aggregate, `O(1)` to rank 3 rows.

In [18]:
mysql_sql = """
SELECT
    c.segment                                   AS segment,
    ROUND(SUM(o.sales), 2)                      AS segment_sales,
    RANK() OVER (ORDER BY SUM(o.sales) DESC)    AS segment_rank
FROM customers AS c
INNER JOIN orders AS o
    ON o.customer_id = c.customer_id
GROUP BY c.segment;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
df = pd.read_sql_query(mysql_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
SELECT
    c.segment                                   AS segment,
    ROUND(SUM(o.sales), 2)                      AS segment_sales,
    RANK() OVER (ORDER BY SUM(o.sales) DESC)    AS segment_rank
FROM customers AS c
INNER JOIN orders AS o
    ON o.customer_id = c.customer_id
GROUP BY c.segment;


Rows returned: 3

    segment  segment_sales  segment_rank
   Consumer     1161401.34             1
  Corporate      706146.37             2
Home Office      429653.15             3


## Insight
With only 3 segments, ranking is more about magnitude than order - the
percentage gap between #1 and #3 usually matters more to a budget decision than the
ordinal rank itself.

# Query 9.4 - Category % Contribution to Total Sales

## Explanation
Concept demonstrated: Subquery + Aggregation

Why this query: shows which product categories actually drive revenue - important for
inventory and merchandising decisions.

Steps: the scalar subquery `(SELECT SUM(sales) FROM orders)` computes the grand
total once; each category's sum is divided by it to get a percentage.
Time complexity: `O(n)` for the grand total, `O(n)` for the per-category sums -
effectively two linear passes.

In [19]:
mysql_sql = """
SELECT
    p.category                                                          AS category,
    ROUND(SUM(o.sales), 2)                                              AS category_sales,
    ROUND(100.0 * SUM(o.sales) / (SELECT SUM(sales) FROM orders), 2)    AS pct_of_total_sales
FROM products AS p
INNER JOIN orders AS o
    ON o.product_id = p.product_id
GROUP BY p.category
ORDER BY category_sales DESC;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
df = pd.read_sql_query(mysql_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
SELECT
    p.category                                                          AS category,
    ROUND(SUM(o.sales), 2)                                              AS category_sales,
    ROUND(100.0 * SUM(o.sales) / (SELECT SUM(sales) FROM orders), 2)    AS pct_of_total_sales
FROM products AS p
INNER JOIN orders AS o
    ON o.product_id = p.product_id
GROUP BY p.category
ORDER BY category_sales DESC;


Rows returned: 3

       category  category_sales  pct_of_total_sales
     Technology       836154.03                36.4
      Furniture       741999.80                32.3
Office Supplies       719047.03                31.3


## Insight
If one category dominates the mix, the business is concentrated in a single
product line - worth flagging as a concentration risk alongside the customer-concentration
finding in the main report.

# Query 9.5 - Order Value Segmentation

## Explanation
Concept demonstrated: CASE Statement

Why this query: turns a continuous `sales` number into a business-readable category,
the sort of transformation dashboards and BI tools rely on constantly.

Steps: `CASE` evaluates its `WHEN` clauses top to bottom for every row and
returns the first matching label (`High Value` ≥ 1000, else `Medium Value` ≥ 300, else
`Low Value`).
Time complexity: `O(n)` - one pass, constant-time branching per row.

In [20]:
mysql_sql = """
SELECT
    order_id,
    sales,
    CASE
        WHEN sales >= 1000 THEN 'High Value'
        WHEN sales >= 300  THEN 'Medium Value'
        ELSE 'Low Value'
    END AS value_segment
FROM orders
ORDER BY sales DESC;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
df = pd.read_sql_query(mysql_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.head(15).to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
SELECT
    order_id,
    sales,
    CASE
        WHEN sales >= 1000 THEN 'High Value'
        WHEN sales >= 300  THEN 'Medium Value'
        ELSE 'Low Value'
    END AS value_segment
FROM orders
ORDER BY sales DESC;


Rows returned: 9994

      order_id     sales value_segment
CA-2014-145317 22638.480    High Value
CA-2016-118689 17499.950    High Value
CA-2017-140151 13999.960    High Value
CA-2017-127180 11199.968    High Value
CA-2017-166709 10499.970    High Value
CA-2016-117121  9892.740    High Value
CA-2014-116904  9449.950    High Value
US-2016-107440  9099.930    High Value
CA-2016-158841  8749.950    High Value
CA-2016-143714  8399.976    High Value
CA-2014-143917  8187.650    High Value
CA-2014-139892  8159.952    High Value
US-2017-168116  7999.980    High Value
CA-2014-145541  6999.960    High Value
CA-2015-145352  6354.950    High Value


## Insight
The thresholds (1000 / 300) are just examples - in a real assignment they should
be set from the actual sales distribution (e.g. percentiles) rather than round numbers, so
the three buckets end up reasonably populated instead of one dominating.

# Query 9.6 - Running Total of Sales per Customer

## Explanation
Concept demonstrated: Window Function: SUM() OVER (cumulative)

Why this query: shows the trajectory of a customer's spend over time, not just the
final total - useful for spotting when a customer's cumulative value crossed a loyalty
threshold.

Steps: `PARTITION BY customer_id` isolates each customer's rows;
`ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` tells the window frame to include every
row from the start of that customer's history up to the current one, summing as it goes.
Time complexity: `O(n log n)` to sort within partitions, `O(n)` to compute the running
sums in a single pass thereafter.

In [21]:
mysql_sql = """
SELECT
    c.customer_name AS customer_name,
    o.order_date    AS order_date,
    o.sales         AS sales,
    ROUND(
        SUM(o.sales) OVER (
            PARTITION BY o.customer_id
            ORDER BY o.order_date, o.row_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ), 2
    ) AS running_total_sales
FROM orders AS o
INNER JOIN customers AS c
    ON c.customer_id = o.customer_id
ORDER BY c.customer_name, o.order_date;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
df = pd.read_sql_query(mysql_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.head(15).to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
SELECT
    c.customer_name AS customer_name,
    o.order_date    AS order_date,
    o.sales         AS sales,
    ROUND(
        SUM(o.sales) OVER (
            PARTITION BY o.customer_id
            ORDER BY o.order_date, o.row_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ), 2
    ) AS running_total_sales
FROM orders AS o
INNER JOIN customers AS c
    ON c.customer_id = o.customer_id
ORDER BY c.customer_name, o.order_date;


Rows returned: 9994

customer_name order_date   sales  running_total_sales
Aaron Bergman 2014-02-18  12.624                12.62
Aaron Bergman 2014-03-07  48.712                61.34
Aaron Bergman 2014-03-07  17.940                79.28
Aaron Bergman 2014-03-07 242.940               322.22
Aaron Bergman 2016-11-10 221.980               544.20
Aaron Bergman 2016-11-10 341.960               886.16
Aaron Hawkins 2014-04-22 247.840               247.84
Aaron Hawkins 2014-

## Insight
The final running-total row for each customer equals their `total_sales` from
Query 2.3 - a decent way to double check that the window frame is defined correctly.

# Query 9.7 - Customers with Declining Sales (First Half vs. Second Half)

## Explanation
Concept demonstrated: CTE + CASE + Aggregation

Why this query: flags customers whose buying has slowed in the second half of the
observed period vs. the first - an early warning sign for churn.

Steps:
1. `bounds` finds the overall first and last order date in the dataset.
2. `half_sales` splits each customer's sales into "before the midpoint" and "at/after the
 midpoint" using conditional `SUM(CASE WHEN ...)` aggregation.
3. A final `CASE` labels each customer Declining / Growing / Flat by comparing the two
 halves.

Time complexity: `O(n)` to scan and bucket `n` order lines into two conditional sums,
`O(m log m)` to sort `m` customers by the resulting difference.

In [22]:
mysql_sql = """
WITH bounds AS (
    SELECT MIN(order_date) AS min_d, MAX(order_date) AS max_d FROM orders
),
half_sales AS (
    SELECT
        o.customer_id,
        SUM(CASE WHEN o.order_date <  DATE_ADD(b.min_d, INTERVAL DATEDIFF(b.max_d, b.min_d)/2 DAY)
                 THEN o.sales ELSE 0 END) AS first_half_sales,
        SUM(CASE WHEN o.order_date >= DATE_ADD(b.min_d, INTERVAL DATEDIFF(b.max_d, b.min_d)/2 DAY)
                 THEN o.sales ELSE 0 END) AS second_half_sales
    FROM orders AS o
    CROSS JOIN bounds AS b
    GROUP BY o.customer_id
)
SELECT
    c.customer_name                          AS customer_name,
    ROUND(hs.first_half_sales, 2)            AS first_half_sales,
    ROUND(hs.second_half_sales, 2)           AS second_half_sales,
    CASE
        WHEN hs.second_half_sales < hs.first_half_sales THEN 'Declining'
        WHEN hs.second_half_sales > hs.first_half_sales THEN 'Growing'
        ELSE 'Flat'
    END AS trend
FROM half_sales AS hs
INNER JOIN customers AS c ON c.customer_id = hs.customer_id
ORDER BY (hs.second_half_sales - hs.first_half_sales) ASC;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
print("[Executed via an equivalent SQLite-compatible query in this sandbox notebook -- see note below]")
exec_sql = """
WITH bounds AS (
    SELECT MIN(order_date) AS min_d, MAX(order_date) AS max_d FROM orders
),
half_sales AS (
    SELECT
        o.customer_id,
        SUM(CASE WHEN o.order_date < date(b.min_d, '+' || CAST((julianday(b.max_d) - julianday(b.min_d)) / 2 AS INT) || ' days')
                 THEN o.sales ELSE 0 END) AS first_half_sales,
        SUM(CASE WHEN o.order_date >= date(b.min_d, '+' || CAST((julianday(b.max_d) - julianday(b.min_d)) / 2 AS INT) || ' days')
                 THEN o.sales ELSE 0 END) AS second_half_sales
    FROM orders AS o
    CROSS JOIN bounds AS b
    GROUP BY o.customer_id
)
SELECT
    c.customer_name                          AS customer_name,
    ROUND(hs.first_half_sales, 2)            AS first_half_sales,
    ROUND(hs.second_half_sales, 2)           AS second_half_sales,
    CASE
        WHEN hs.second_half_sales < hs.first_half_sales THEN 'Declining'
        WHEN hs.second_half_sales > hs.first_half_sales THEN 'Growing'
        ELSE 'Flat'
    END AS trend
FROM half_sales AS hs
INNER JOIN customers AS c ON c.customer_id = hs.customer_id
ORDER BY (hs.second_half_sales - hs.first_half_sales) ASC;
"""
df = pd.read_sql_query(exec_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.head(15).to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
WITH bounds AS (
    SELECT MIN(order_date) AS min_d, MAX(order_date) AS max_d FROM orders
),
half_sales AS (
    SELECT
        o.customer_id,
        SUM(CASE WHEN o.order_date <  DATE_ADD(b.min_d, INTERVAL DATEDIFF(b.max_d, b.min_d)/2 DAY)
                 THEN o.sales ELSE 0 END) AS first_half_sales,
        SUM(CASE WHEN o.order_date >= DATE_ADD(b.min_d, INTERVAL DATEDIFF(b.max_d, b.min_d)/2 DAY)
                 THEN o.sales ELSE 0 END) AS second_half_sales
    FROM orders AS o
    CROSS JOIN bounds AS b
    GROUP BY o.customer_id
)
SELECT
    c.customer_name                          AS customer_name,
    ROUND(hs.first_half_sales, 2)            AS first_half_sales,
    ROUND(hs.second_half_sales, 2)           AS second_half_sales,
    CASE
        WHEN hs.second_half_sales < hs.first_half_sales THEN 'Declining'
        WHEN hs.second_half_sales > hs.first_half_sales THEN 'Growing'
        ELSE 'Flat'
    END 

> **Dialect note:** MySQL's `DATE_ADD(date, INTERVAL n DAY)` / `DATEDIFF(d1, d2)` become SQLite's `date(date, '+n days')` / `julianday(d1) - julianday(d2)` - same midpoint-of-range calculation, different function names.

## Insight
This is a coarse trend signal (two buckets only) - a real churn-risk model
would use more periods and a statistical trend test, but the two-bucket version is a fast,
readable first pass that's easy to explain to a non-technical stakeholder.

# Query 9.8 - Best-Selling Product per Category

## Explanation
Concept demonstrated: Window Function + PARTITION BY

Why this query: identifies the flagship product driving revenue within each product
category - useful for merchandising and restocking priorities.

Steps:
1. `product_sales` totals sales per product within its category.
2. `RANK() OVER (PARTITION BY category ORDER BY product_sales DESC)` restarts the ranking
 at 1 for every category (that's what `PARTITION BY` does, as in Query 2.6).
3. The outer query keeps only rank-1 products - the category leader(s).

Time complexity: `O(n)` to aggregate order lines into per-product sales, `O(p log p)`
to sort `p` products within their category partitions.

In [23]:
mysql_sql = """
WITH product_sales AS (
    SELECT
        p.category,
        p.product_name,
        SUM(o.sales) AS product_sales
    FROM products AS p
    INNER JOIN orders AS o ON o.product_id = p.product_id
    GROUP BY p.category, p.product_name
),
ranked_products AS (
    SELECT
        category,
        product_name,
        ROUND(product_sales, 2) AS product_sales,
        RANK() OVER (PARTITION BY category ORDER BY product_sales DESC) AS rank_in_category
    FROM product_sales
)
SELECT * FROM ranked_products WHERE rank_in_category = 1;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
df = pd.read_sql_query(mysql_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
WITH product_sales AS (
    SELECT
        p.category,
        p.product_name,
        SUM(o.sales) AS product_sales
    FROM products AS p
    INNER JOIN orders AS o ON o.product_id = p.product_id
    GROUP BY p.category, p.product_name
),
ranked_products AS (
    SELECT
        category,
        product_name,
        ROUND(product_sales, 2) AS product_sales,
        RANK() OVER (PARTITION BY category ORDER BY product_sales DESC) AS rank_in_category
    FROM product_sales
)
SELECT * FROM ranked_products WHERE rank_in_category = 1;


Rows returned: 3

       category                                                                product_name  product_sales  rank_in_category
      Furniture                                HON 5400 Series Task Chairs for Big and Tall       21870.58                 1
Office Supplies Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind       27453.38               

## Insight
Because `RANK()` (not `ROW_NUMBER()`) is used, a category with two products
exactly tied for the top spot will correctly return both - swapping in `ROW_NUMBER()` here
would silently drop one of them.

# Query 9.9 - Average Discount by Region, Ranked

## Explanation
Concept demonstrated: Aggregation + RANK

Why this query: heavy discounting erodes margin - this flags which regions are
discounting most aggressively, a typical finance/ops review question.

Steps: average the `discount` fraction per region directly on `orders` (region
is a ship-to attribute of the order, not a fixed customer attribute - see the Data Setup
section), then rank regions by that average.
Time complexity: `O(n)` aggregate over `n` order lines, `O(1)` to rank a handful of
regions.

In [24]:
mysql_sql = """
-- region lives on `orders` (ship-to location per order), not `customers` -- see Data Setup
SELECT
    o.region                                            AS region,
    ROUND(AVG(o.discount), 3)                           AS avg_discount,
    RANK() OVER (ORDER BY AVG(o.discount) DESC)         AS discount_rank
FROM orders AS o
GROUP BY o.region;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
df = pd.read_sql_query(mysql_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
-- region lives on `orders` (ship-to location per order), not `customers` -- see Data Setup
SELECT
    o.region                                            AS region,
    ROUND(AVG(o.discount), 3)                           AS avg_discount,
    RANK() OVER (ORDER BY AVG(o.discount) DESC)         AS discount_rank
FROM orders AS o
GROUP BY o.region;


Rows returned: 4

 region  avg_discount  discount_rank
Central         0.240              1
  South         0.147              2
   East         0.145              3
   West         0.109              4


## Insight
A region with an unusually high average discount is worth investigating in
combination with its profit margin (not shown here) - heavy discounting is only a problem
if it isn't paying for itself in volume.

# Query 9.10 - Customer Tenure (Days Between First and Last Order)

## Explanation
Concept demonstrated: Date Functions + Aggregation

Why this query: distinguishes long-standing customers (wide date range, possibly many
orders) from brand-new ones (narrow or zero range) - tenure is a classic input to
lifetime-value modeling.

Steps: group by customer, take the earliest and latest `order_date`, subtract
them for a day count, and count distinct orders in the same pass.
Time complexity: `O(n)` - a single grouped aggregation pass over `n` order lines.

In [25]:
mysql_sql = """
SELECT
    c.customer_name                              AS customer_name,
    MIN(o.order_date)                            AS first_order_date,
    MAX(o.order_date)                            AS last_order_date,
    DATEDIFF(MAX(o.order_date), MIN(o.order_date)) AS tenure_days,
    COUNT(DISTINCT o.order_id)                   AS total_orders
FROM customers AS c
INNER JOIN orders AS o ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.customer_name
ORDER BY tenure_days DESC;
"""
print("-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --")
print(mysql_sql.strip())
print()
print("[Executed via an equivalent SQLite-compatible query in this sandbox notebook -- see note below]")
exec_sql = """
SELECT
    c.customer_name                              AS customer_name,
    MIN(o.order_date)                            AS first_order_date,
    MAX(o.order_date)                            AS last_order_date,
    CAST(julianday(MAX(o.order_date)) - julianday(MIN(o.order_date)) AS INTEGER) AS tenure_days,
    COUNT(DISTINCT o.order_id)                   AS total_orders
FROM customers AS c
INNER JOIN orders AS o ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.customer_name
ORDER BY tenure_days DESC;
"""
df = pd.read_sql_query(exec_sql, conn)
print(f"\nRows returned: {len(df)}\n")
print(df.head(15).to_string(index=False))

-- MySQL 8.0 query (as it appears in superstore_analysis_mysql.sql) --
SELECT
    c.customer_name                              AS customer_name,
    MIN(o.order_date)                            AS first_order_date,
    MAX(o.order_date)                            AS last_order_date,
    DATEDIFF(MAX(o.order_date), MIN(o.order_date)) AS tenure_days,
    COUNT(DISTINCT o.order_id)                   AS total_orders
FROM customers AS c
INNER JOIN orders AS o ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.customer_name
ORDER BY tenure_days DESC;

[Executed via an equivalent SQLite-compatible query in this sandbox notebook -- see note below]

Rows returned: 793

    customer_name first_order_date last_order_date  tenure_days  total_orders
    Michael Moore       2014-01-13      2017-12-23         1440            11
Delfina Latchford       2014-01-16      2017-12-08         1422             8
  Chris Selesnick       2014-01-13      2017-12-04         1421            12
Natalie DeC

> **Dialect note:** MySQL's `DATEDIFF(d1, d2)` becomes SQLite's `julianday(d1) - julianday(d2)` (cast to integer) - identical day-count result.

## Insight
Tenure alone can mislead: a customer with a long tenure but few orders (see
Query MP.3's single-order customers, tenure = 0 by definition) isn't necessarily more
valuable than a short-tenure customer who has already ordered frequently - tenure and
order frequency should be read together.

# Lessons From Running This on a Real MySQL 8.0 Server

The queries above were validated in this notebook against a SQLite copy of the data, but
`superstore_analysis_mysql.sql` was also run end-to-end against a real, local MySQL 8.0
instance (MySQL Workbench) with the real Kaggle CSV. Two real issues surfaced that are
worth recording, since neither would show up testing against a small or synthetic sample.

**1. MySQL Workbench's Table Data Import Wizard silently dropped rows.** Loading
`superstore_raw.csv` through the Wizard's GUI produced a table with 9,694 rows instead of
9,994 -- 300 short, with no error shown. Every missing row had a product name containing
an embedded quotation mark (an inches symbol like `11"`, or a quoted phrase like `"While
You Were Out"`). The CSV correctly escapes these per RFC4180 (a doubled `""` represents
one literal `"`), but the Wizard's parser doesn't unescape that correctly and drops the
row instead of erroring. The fix was to bypass the Wizard entirely and use
`LOAD DATA LOCAL INFILE` with `ESCAPED BY '"'` (using the quote character as its own
escape character), which parses the doubled-quote convention correctly. **Lesson:** a
"successful" GUI import isn't proof of a complete import -- always verify with
`SELECT COUNT(*)` against the known expected row count.

**2. The correlated-subquery version of the `products` insert was unusably slow at real
scale.** The subquery in Part 1.3 that picks a canonical name for the 32 duplicate-name
product IDs re-scans `superstore_raw` once per candidate row with no supporting index --
fine on a small dataset, but on the real 9,994-row table it ran for 200+ seconds and had
to be killed via `SHOW PROCESSLIST` / `KILL <id>`. Rewriting it as a single `GROUP BY
product_id, category, sub_category` with `MAX(product_name)` produces the identical
result in a fraction of a second, because it's one aggregation pass instead of one
subquery execution per row. **Lesson:** a correlated subquery that looks fine in a small
test can hide an O(n²) cost that only shows up at real data volumes -- `EXPLAIN` (or just
watching the clock) before assuming a working query is a fast query.

Both fixes are reflected in the current version of `superstore_analysis_mysql.sql`.

# Summary

This notebook covered, in order: normalized table creation from a raw staging table
(`SELECT DISTINCT`), non-correlated and correlated subqueries, CTEs (standalone and
stacked), the core window functions (`RANK`, `DENSE_RANK`, `ROW_NUMBER`, `LAG`,
cumulative `SUM() OVER`), a combined JOIN+CTE+Window final report, five mini-project
business questions, and ten bonus questions adding date functions and `CASE` logic.

See `REPORT.md` for the condensed write-up (objective, schema, approach, and
consolidated business insights) suitable for direct submission.